In [1]:
import pandas as pd
import pickle
from tqdm.notebook import tqdm

from config import DATA_DIR, WORKING_DIR, Path

# Decompress ClassyFire

In [ ]:
import zstandard as zstd

input_path = 'data/classyfire.v4.tsv.zst' # -> 2.1 GB
output_path = 'data/classyfire.v4.tsv'    # -> ~24 GB

file_size = __import__('os').path.getsize(input_path)
    
decomp = zstd.ZstdDecompressor()

with open(input_path, 'rb') as f_in, \
        open(output_path, 'wb') as f_out, \
        decomp.stream_reader(f_in) as reader, \
        tqdm(total=file_size, unit='B', unit_scale=True, desc="Decompressing") as pbar:
    
    while True:
        chunk = reader.read(1024*1024)
        while chunk:
            f_out.write(chunk)
            pbar.update(len(chunk))

print("Done!")

# Filter MassSpecGym dataset

## Overview

In [2]:
msg_df = pd.read_csv('data/MassSpecGym1.5.tsv', sep='\t')
print(msg_df.shape)
msg_df.info()

(231104, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 231104 entries, 0 to 231103
Data columns (total 14 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   identifier            231104 non-null  object 
 1   mzs                   231104 non-null  object 
 2   intensities           231104 non-null  object 
 3   smiles                231104 non-null  object 
 4   inchikey              231104 non-null  object 
 5   formula               231104 non-null  object 
 6   precursor_formula     231104 non-null  object 
 7   parent_mass           231104 non-null  float64
 8   precursor_mz          231104 non-null  float64
 9   adduct                231104 non-null  object 
 10  instrument_type       225881 non-null  object 
 11  collision_energy      121746 non-null  float64
 12  fold                  231104 non-null  object 
 13  simulation_challenge  231104 non-null  bool   
dtypes: bool(1), float64(3), object(10)
memo

In [3]:
msg_df['fold'].value_counts()

fold
train    194119
val       19429
test      17556
Name: count, dtype: int64

## Processing

In [4]:
working_msg_df = msg_df[msg_df['fold'] == WORKING_DIR]
print(working_msg_df.shape)
working_msg_df.info()

(194119, 14)
<class 'pandas.core.frame.DataFrame'>
Index: 194119 entries, 0 to 231094
Data columns (total 14 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   identifier            194119 non-null  object 
 1   mzs                   194119 non-null  object 
 2   intensities           194119 non-null  object 
 3   smiles                194119 non-null  object 
 4   inchikey              194119 non-null  object 
 5   formula               194119 non-null  object 
 6   precursor_formula     194119 non-null  object 
 7   parent_mass           194119 non-null  float64
 8   precursor_mz          194119 non-null  float64
 9   adduct                194119 non-null  object 
 10  instrument_type       189791 non-null  object 
 11  collision_energy      101609 non-null  float64
 12  fold                  194119 non-null  object 
 13  simulation_challenge  194119 non-null  bool   
dtypes: bool(1), float64(3), object(10)
memory us

In [5]:
working_msg_df['adduct'].value_counts()

adduct
[M+H]+     166479
[M+Na]+     27640
Name: count, dtype: int64

In [6]:
working_msg_df = working_msg_df[working_msg_df['adduct'] == '[M+H]+']
print(working_msg_df.shape)
working_msg_df.info()

(166479, 14)
<class 'pandas.core.frame.DataFrame'>
Index: 166479 entries, 0 to 231094
Data columns (total 14 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   identifier            166479 non-null  object 
 1   mzs                   166479 non-null  object 
 2   intensities           166479 non-null  object 
 3   smiles                166479 non-null  object 
 4   inchikey              166479 non-null  object 
 5   formula               166479 non-null  object 
 6   precursor_formula     166479 non-null  object 
 7   parent_mass           166479 non-null  float64
 8   precursor_mz          166479 non-null  float64
 9   adduct                166479 non-null  object 
 10  instrument_type       162976 non-null  object 
 11  collision_energy      99377 non-null   float64
 12  fold                  166479 non-null  object 
 13  simulation_challenge  166479 non-null  bool   
dtypes: bool(1), float64(3), object(10)
memory us

In [7]:
working_msg_df['inchikey'].value_counts()

inchikey
AIONOLUJZLIMTK    415
PFTAWBLQPZVEMU    361
SRIGHEHXEGELQJ    304
NEGQHKSYEYVFTD    303
SULIDBRAXVDKBU    301
                 ... 
RWZHXRGQHNGCQY      1
KARPTPWMCFFTRS      1
ISKBJTAHHJGJCX      1
XAECBOBIXXXVRH      1
UWQOGHWNWIJMNI      1
Name: count, Length: 22022, dtype: int64

In [ ]:
dedupl_msg_df = working_msg_df.drop_duplicates('inchikey')
print(dedupl_msg_df.shape)
dedupl_msg_df.info()

(2899, 14)
<class 'pandas.core.frame.DataFrame'>
Index: 2899 entries, 170 to 222364
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   identifier            2899 non-null   object 
 1   mzs                   2899 non-null   object 
 2   intensities           2899 non-null   object 
 3   smiles                2899 non-null   object 
 4   inchikey              2899 non-null   object 
 5   formula               2899 non-null   object 
 6   precursor_formula     2899 non-null   object 
 7   parent_mass           2899 non-null   float64
 8   precursor_mz          2899 non-null   float64
 9   adduct                2899 non-null   object 
 10  instrument_type       2718 non-null   object 
 11  collision_energy      2264 non-null   float64
 12  fold                  2899 non-null   object 
 13  simulation_challenge  2899 non-null   bool   
dtypes: bool(1), float64(3), object(10)
memory usage: 319.9+ KB


# Enrich MassSpecGym dataset with ClassyFire ChemONT trees

## Create pickle with ChemONT trees of interest

In [8]:
chemont_trees_cache_path = Path(DATA_DIR, WORKING_DIR, "chemont_cache.pkl")

In [ ]:
inchikeys = list(working_msg_df['inchikey'].drop_duplicates().values)
aux_d = {
    'inchikey': dict(),
    'chemont_tree': dict()
    }

for i, k in enumerate(inchikeys):
    aux_d['inchikey'][i] = k
    aux_d['chemont_tree'][i] = None

key_chemont_df = pd.DataFrame.from_dict(aux_d)

In [ ]:
classyfire_dfs = pd.read_csv('data/classyfire.v4.tsv', sep='\t', chunksize=20000)

for df in tqdm(classyfire_dfs):
    df['inchikey'] = df['inchikey'].apply(lambda key: key.split("-")[0])  # Keep only first block of the Inchikey
    df.drop_duplicates('inchikey', inplace=True)
    df = df[['inchikey', 'chemont_tree_json']]

    key_chemont_df = pd.merge(key_chemont_df, df, on='inchikey', how='left')

    key_chemont_df['chemont_tree'].where(key_chemont_df['chemont_tree'].notnull(), key_chemont_df['chemont_tree_json'], inplace=True)

    key_chemont_df.drop('chemont_tree_json', axis=1, inplace=True)

with open(chemont_trees_cache_path, "wb") as temp:
    key_chemont_df.to_pickle(temp)

key_chemont_df.info()
key_chemont_df

0it [00:00, ?it/s]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2998 entries, 0 to 2997
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   inchikey      2998 non-null   object
 1   chemont_tree  2987 non-null   object
dtypes: object(2)
memory usage: 47.0+ KB


,inchikey,chemont_tree
0,GYSCAQFHASJXRS,"[0,264,1813,1961,1994]"
1,OUMWCYMRLMEZJH,"[1,423,453,554,554]"
2,ZRTGPZGAMCJZNA,"[0,12,259,1550,1550]"
3,HWSQVPGTQUYLEQ,"[0,12,3909,1334,2518]"
4,MXLWQNCWIIZUQT,"[0,2448,2903,2903,2903]"
...,...,...
2993,XMSXQFUHVRWGNA,"[0,462,1370,3194,4445]"
2994,HXIQYSLFEXIOAV,"[1,423,453,555,555]"
2995,LYUPEIXJYAJCHL,"[0,12,259,1551,3788]"
2996,PMCAUYATZKXGHU,"[0,12,259,1551,3788]"


In [9]:
with open(chemont_trees_cache_path, "rb") as f:
    key_chemont_df = pickle.load(f)

key_chemont_df[key_chemont_df['chemont_tree'].isnull()]

FileNotFoundError: [Errno 2] No such file or directory: 'data\\train\\chemont_cache.pkl'

## Join MSG dataset with ChemONT trees

In [13]:
target_msg_df = working_msg_df.copy()

result_df = pd.merge(target_msg_df, key_chemont_df, on='inchikey')

result_df.info()
result_df[result_df['chemont_tree'].isnull()]


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14066 entries, 0 to 14065
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   identifier            14066 non-null  object 
 1   mzs                   14066 non-null  object 
 2   intensities           14066 non-null  object 
 3   smiles                14066 non-null  object 
 4   inchikey              14066 non-null  object 
 5   formula               14066 non-null  object 
 6   precursor_formula     14066 non-null  object 
 7   parent_mass           14066 non-null  float64
 8   precursor_mz          14066 non-null  float64
 9   adduct                14066 non-null  object 
 10  instrument_type       13816 non-null  object 
 11  collision_energy      9957 non-null   float64
 12  fold                  14066 non-null  object 
 13  simulation_challenge  14066 non-null  bool   
 14  chemont_tree          14056 non-null  object 
dtypes: bool(1), float64

,identifier,mzs,intensities,smiles,inchikey,formula,precursor_formula,parent_mass,precursor_mz,adduct,instrument_type,collision_energy,fold,simulation_challenge,chemont_tree
13965,MassSpecGymID0397635,"52.847301,68.049599,77.991096,80.049797,95.012...","0.004090909090909091,0.005681818181818182,0.00...",CC(=O)OC1C(OC(=O)c2ccc(=O)n(C)c2)CC(C)C23OC(C)...,YNMMSSYXBYTABV,C37H44N2O14,C37H45N2O14,740.279724,741.287,[M+H]+,Orbitrap,NaN,test,False,NaN
13966,MassSpecGymID0397636,"57.070499,59.049702,70.788002,80.049698,85.064...","0.066,0.0037,0.0037,0.011,0.0074,0.004,0.02,0....",CC[C@@H](C)C(=O)OC[C@]12C(=O)[C@@H](OC(=O)c3cc...,HPZNCFSLZGFDST,C36H44N2O13,C36H45N2O13,712.224724,713.232,[M+H]+,Orbitrap,NaN,test,False,NaN
13967,MassSpecGymID0397637,"57.070499,80.049599,95.013,108.0448,136.039505...","0.024390243902439025,0.008048780487804878,0.01...",CC[C@@H](C)C(=O)OC[C@]12C(=O)C(OC(=O)c3ccc(=O)...,ATASRQYZQYFECN,C38H46N2O14,C38H47N2O14,754.294724,755.302,[M+H]+,Orbitrap,NaN,test,False,NaN
13968,MassSpecGymID0397638,"57.070599,59.049801,85.065102,95.012802,105.07...","0.16756756756756758,0.02,0.023513513513513513,...",CC[C@@H](C)C(=O)OC[C@]12[C@H](OC(=O)[C@@H](C)C...,LVFIUDAMNWXFMK,C41H54N2O14,C41H55N2O14,798.357724,799.365,[M+H]+,Orbitrap,NaN,test,False,NaN
13969,MassSpecGymID0397639,"59.049801,68.050201,71.049698,80.050102,83.049...","0.00411764705882353,0.004294117647058823,0.011...",CC[C@H](C)C(=O)O[C@@H]1[C@H](OC(=O)c2ccc(=O)n(...,HDNFKWOIOAMTET,C42H54N2O15,C42H55N2O15,826.352724,827.360,[M+H]+,Orbitrap,NaN,test,False,NaN
13976,MassSpecGymID0397815,"55.019798,76.023003,84.044502,130.050201,131.0...","0.04905660377358491,0.09056603773584905,0.1660...",N[C@@H](CCC(=O)N[C@H]1CSSC[C@@H](C(=O)NCC(=O)O...,LGSJLBWWZAOHIV,C18H27N5O10S2,C18H28N5O10S2,537.122724,538.130,[M+H]+,QTOF,NaN,test,False,NaN
13983,MassSpecGymID0397912,"51.695801,54.804001,54.806301,56.807701,56.918...","0.0033333333333333335,0.0021538461538461538,0....",CCC1=C(C)C2=NC1=CC1=C(C)C3=C(O)[C@H](O)C(=C4NC...,NXPWZDQBJBVYGK,C32H34N4O5,C32H35N4O5,554.252724,555.260,[M+H]+,Orbitrap,NaN,test,False,NaN
13988,MassSpecGymID0397917,"50.022499,51.577702,51.5798,51.6917,53.390598,...","0.0013225806451612903,0.0013548387096774194,0....",CCC1=C(C)C2=NC1=CC1=C(C)C3=C(O)[C@H](C(=O)OC)C...,IQZKGDLOGOUDKG,C34H36N4O6,C34H37N4O6,596.262724,597.270,[M+H]+,Orbitrap,NaN,test,False,NaN
13989,MassSpecGymID0397948,"123.025902,123.066399,124.0336,138.050293,151....","0.024,0.15,0.014,0.013,0.015,0.008,0.049,0.011...",COc1cc(C(Cc2c(O)c(SC)c(SC)c(O)c2SC)c2cnc(N)nc2...,GOMIKTMVKFBMCU,C24H30N4O5S3,C24H31N4O5S3,550.137724,551.145,[M+H]+,QTOF,NaN,test,False,NaN
14005,MassSpecGymID0398064,"81.069809,93.069672,95.049347,95.085617,105.07...","0.039912263326107665,0.069775834152626,0.18805...",CC(=O)OC[C@]12CCC3C(CC=C4CC=CC(=O)[C@@]43C)[C@...,YDHYIMYYVRIJQZ,C38H52O16,C38H53O16,764.325724,765.333,[M+H]+,Orbitrap,NaN,test,False,NaN


## Format ChemONT tree column

In [17]:
from ast import literal_eval
    
# Function to parse list-like strings into real lists
def format_chemont_tree(df: pd.DataFrame, chemont_column : str = 'chemont_tree'):
    for index, row in df[df[chemont_column].notnull()].iterrows():
        tree = row[chemont_column]
        try:
            new_tree = literal_eval(tree)
        except ValueError:
            if "null" in tree:
                new_tree = literal_eval(tree.replace("null", "None"))

        df.at[index, chemont_column] = new_tree

format_chemont_tree(result_df)

In [18]:
result_df.info()
result_df

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14066 entries, 0 to 14065
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   identifier            14066 non-null  object 
 1   mzs                   14066 non-null  object 
 2   intensities           14066 non-null  object 
 3   smiles                14066 non-null  object 
 4   inchikey              14066 non-null  object 
 5   formula               14066 non-null  object 
 6   precursor_formula     14066 non-null  object 
 7   parent_mass           14066 non-null  float64
 8   precursor_mz          14066 non-null  float64
 9   adduct                14066 non-null  object 
 10  instrument_type       13816 non-null  object 
 11  collision_energy      9957 non-null   float64
 12  fold                  14066 non-null  object 
 13  simulation_challenge  14066 non-null  bool   
 14  chemont_tree          14056 non-null  object 
dtypes: bool(1), float64

,identifier,mzs,intensities,smiles,inchikey,formula,precursor_formula,parent_mass,precursor_mz,adduct,instrument_type,collision_energy,fold,simulation_challenge,chemont_tree
0,MassSpecGymID0000203,"134.0964,234.1462,244.1305,262.1411","1.0,0.07907907907907907,0.3733733733733734,0.1...",CC(C)[C@H]1OC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C@@...,GYSCAQFHASJXRS,C45H57N3O9,C45H58N3O9,783.408924,784.4162,[M+H]+,Orbitrap,35.0,test,True,"[0, 264, 1813, 1961, 1994]"
1,MassSpecGymID0000204,"91.0542,134.0964,244.1305","0.043043043043043044,1.0,0.04804804804804805",CC(C)[C@H]1OC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C@@...,GYSCAQFHASJXRS,C45H57N3O9,C45H58N3O9,783.408924,784.4162,[M+H]+,Orbitrap,50.0,test,True,"[0, 264, 1813, 1961, 1994]"
2,MassSpecGymID0000205,"134.0964,216.1356,234.1462,244.1305,262.1411,3...","1.0,0.037037037037037035,0.15615615615615616,0...",CC(C)[C@H]1OC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C@@...,GYSCAQFHASJXRS,C45H57N3O9,C45H58N3O9,783.408924,784.4162,[M+H]+,Orbitrap,30.0,test,True,"[0, 264, 1813, 1961, 1994]"
3,MassSpecGymID0000206,"134.0964,234.1462,244.1305,262.1411,362.1935,5...","0.2072072072072072,0.11311311311311312,1.0,0.6...",CC(C)[C@H]1OC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C@@...,GYSCAQFHASJXRS,C45H57N3O9,C45H58N3O9,783.408924,784.4162,[M+H]+,Orbitrap,20.0,test,True,"[0, 264, 1813, 1961, 1994]"
4,MassSpecGymID0000208,"244.1332,262.1411,541.289,784.4168","0.5185185185185185,0.2722722722722723,0.085085...",CC(C)[C@H]1OC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C@@...,GYSCAQFHASJXRS,C45H57N3O9,C45H58N3O9,783.408924,784.4162,[M+H]+,Orbitrap,10.0,test,True,"[0, 264, 1813, 1961, 1994]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14061,MassSpecGymID0400854,"55.055092,57.07082,58.024075,61.011219,62.9905...","0.0022286420583862614,0.01249259525147176,0.00...",C[Si]1(C)O[Si](C)(C)O[Si](C)(C)O[Si](C)(C)O[Si...,XMSXQFUHVRWGNA,C10H30O5Si5,C10H31O5Si5,370.094724,371.1020,[M+H]+,QTOF,NaN,test,False,"[0, 462, 1370, 3194, 4445]"
14062,MassSpecGymID0401425,"53.188965,55.054928,56.187614,57.067844,57.070...","0.0014692404421424558,0.0075411048667320514,0....",Cc1cc(O)c(C(C)(C)C)cc1Sc1cc(C(C)(C)C)c(O)cc1C,HXIQYSLFEXIOAV,C22H30O2S,C22H31O2S,358.192724,359.2000,[M+H]+,QTOF,NaN,test,False,"[1, 423, 453, 555, 555]"
14063,MassSpecGymID0402033,"135.043198,249.128204,263.141907,265.156799,28...","6.487956589565239e-05,3.5798763969405303e-06,3...",CCN1CC2(COC)CCC(OC)C34C5CC6C(OC)CC(OC(C)=O)(C5...,LYUPEIXJYAJCHL,C35H49NO9,C35H50NO9,627.342724,628.3500,[M+H]+,QTOF,NaN,test,False,"[0, 12, 259, 1551, 3788]"
14064,MassSpecGymID0402039,"102.090202,108.079903,154.121597,250.179199,25...","5.842000366504345e-06,1.7871895579441808e-06,0...",CCN1C[C@]2(COC)CC[C@H](O)[C@]34C1[C@@H]([C@H](...,PMCAUYATZKXGHU,C32H45NO8,C32H46NO8,571.317724,572.3250,[M+H]+,QTOF,NaN,test,False,"[0, 12, 259, 1551, 3788]"


In [21]:
result_df.to_csv("data/MSG_test_fold_w_chemont.tsv", sep="\t", index=False)